# Overfitting: When a Model Memorizes Instead of Learns

## One-hour machine learning case study

A marketing team builds a model to predict customer response. A deep decision tree achieves 100% training accuracy but performs poorly on new customers.

This notebook explores underfitting, overfitting, model complexity, validation, and cross-validation.

> **Central question:** Is the model learning a general pattern, or memorizing noise in the training sample?

## Learning objectives

Students will:

- distinguish training performance from generalization;
- recognize underfitting and overfitting;
- compare shallow and deep decision trees;
- use validation curves and cross-validation;
- explain the bias-variance trade-off in practical language;
- avoid choosing a model based on test-set performance;
- use AI to challenge model-selection reasoning;
- recommend a defensible complexity level.

## Suggested schedule

| Time | Activity |
|---|---|
| 0–7 min | React to 100% training accuracy |
| 7–17 min | Build the synthetic customer dataset |
| 17–29 min | Compare tree depths |
| 29–39 min | Use cross-validation |
| 39–47 min | Examine noise and sample size |
| 47–53 min | Use AI as a model-review partner |
| 53–58 min | Transfer to another ML problem |
| 58–60 min | Reflection |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X, y = make_classification(
    n_samples=1200,
    n_features=12,
    n_informative=5,
    n_redundant=2,
    n_repeated=0,
    n_clusters_per_class=2,
    flip_y=0.12,
    class_sep=0.8,
    random_state=42
)

feature_names = [f"feature_{i+1}" for i in range(X.shape[1])]
X = pd.DataFrame(X, columns=feature_names)
y = pd.Series(y, name="responded")

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.25,
    random_state=42, stratify=y_train_full
)

X.shape, X_train.shape, X_valid.shape, X_test.shape

# Part 1 — Initial claim

A tree with unlimited depth reaches 100% training accuracy.

Before testing it, answer:

1. Is 100% training accuracy always desirable?
2. What could the model be memorizing?
3. Which dataset should be used for model selection?
4. When should the test set be used?

In [ ]:
deep_tree = DecisionTreeClassifier(random_state=42)
deep_tree.fit(X_train, y_train)

train_acc = accuracy_score(y_train, deep_tree.predict(X_train))
valid_acc = accuracy_score(y_valid, deep_tree.predict(X_valid))

pd.Series({
    "Training accuracy": train_acc,
    "Validation accuracy": valid_acc
}).to_frame("Unlimited-depth tree").style.format("{:.1%}")

# Part 2 — Compare model complexity

Train decision trees with different maximum depths.

In [ ]:
rows = []

for depth in range(1, 26):
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    rows.append({
        "Depth": depth,
        "Training accuracy": accuracy_score(y_train, model.predict(X_train)),
        "Validation accuracy": accuracy_score(y_valid, model.predict(X_valid))
    })

depth_results = pd.DataFrame(rows)
depth_results.head()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(depth_results["Depth"], depth_results["Training accuracy"], marker="o", label="Training")
plt.plot(depth_results["Depth"], depth_results["Validation accuracy"], marker="o", label="Validation")
plt.xlabel("Maximum tree depth")
plt.ylabel("Accuracy")
plt.title("Training and validation performance by tree depth")
plt.ylim(0.5, 1.02)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## Interpretation

Identify:

- a depth that underfits;
- a depth range that generalizes reasonably;
- a depth range that overfits;
- why training accuracy continues to improve while validation accuracy does not.

In [ ]:
best_validation_row = depth_results.loc[depth_results["Validation accuracy"].idxmax()]
best_validation_row

# Part 3 — Why one validation split is not enough

A single validation split can be lucky or unlucky. Cross-validation repeats the evaluation across several training-validation partitions.

In [ ]:
cv_rows = []

for depth in range(1, 21):
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    scores = cross_val_score(
        model, X_train_full, y_train_full,
        scoring="accuracy", cv=5
    )
    cv_rows.append({
        "Depth": depth,
        "Mean CV accuracy": scores.mean(),
        "CV standard deviation": scores.std()
    })

cv_results = pd.DataFrame(cv_rows)
cv_results.head()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(cv_results["Depth"], cv_results["Mean CV accuracy"], marker="o")
plt.xlabel("Maximum tree depth")
plt.ylabel("Mean cross-validation accuracy")
plt.title("Cross-validation performance by model complexity")
plt.ylim(0.5, 1.0)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
best_depth = int(
    cv_results.loc[cv_results["Mean CV accuracy"].idxmax(), "Depth"]
)

final_model = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
final_model.fit(X_train_full, y_train_full)

test_accuracy = accuracy_score(y_test, final_model.predict(X_test))

pd.Series({
    "Selected depth": best_depth,
    "Final test accuracy": test_accuracy
})

## The role of the test set

The test set should answer one final question:

> How well does the selected modeling process perform on untouched data?

It should not be repeatedly used to choose depth, features, hyperparameters, or preprocessing.

# Part 4 — Bias and variance in plain language

- **Underfitting:** The model is too simple to capture important structure.
- **Overfitting:** The model captures training-specific noise that does not repeat.
- **Good generalization:** The model captures patterns stable enough to work on new data.

Explain how the graph demonstrates all three.

## AI as a critical reviewer

After selecting a depth yourself, use one prompt:

> Challenge my decision-tree depth selection. Ask what evidence I used.

> What signs in the learning curve suggest overfitting?

> Why is repeatedly checking the test set a form of leakage?

> What additional evaluation would you request before deployment?

Document:

**Useful challenge:**  

**Point requiring verification:**  

**Change to my reasoning:**

# Transfer task

A neural network reaches:

- 99% training accuracy;
- 72% validation accuracy;
- 71% test accuracy.

Answer:

1. What is the likely problem?
2. What evidence supports that diagnosis?
3. Name three possible responses.
4. Which dataset should guide tuning?
5. Why is adding more layers not automatically helpful?

# Exit reflection

- Training performance measures …
- Validation performance measures …
- The test set should be used …
- A complex model is justified only when …
- Cross-validation helps because …

# Instructor checklist

- [ ] Students compared training and validation curves.
- [ ] Students identified underfitting and overfitting.
- [ ] Students used cross-validation for model selection.
- [ ] Students preserved the test set for final evaluation.
- [ ] AI challenged the reasoning rather than selected the model.
- [ ] Students transferred the concept to another algorithm.

# Closing principle

The best model is not the one that remembers the training data most accurately.

It is the one that performs reliably on cases it has never seen.